# Stage 1 GRPO Training Scaffold

Minimal Hugging Face TRL training path for the Runway Zero operations-only stage. Run this in Colab or onsite GPU compute.

In [ ]:
!pip install -e .[training]
from datasets import Dataset
from scripts.train_grpo_stage1 import build_prompts, runway_reward
prompts = build_prompts(stage=1)
dataset = Dataset.from_dict({'prompt': prompts})
len(dataset), dataset[0]['prompt'][:500]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

model_name = 'Qwen/Qwen2.5-Coder-7B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')

config = GRPOConfig(
    output_dir='runway-zero-stage1-grpo',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    logging_steps=1,
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=dataset,
    reward_funcs=runway_reward,
)
# trainer.train()